# Análise de Resultados dos Experimentos

In [30]:
import os
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Configurações globais de exibição do Pandas
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", "{:.6f}".format)

In [31]:
# Cache para evitar a recarga de arquivos de fronteiras de referência repetidamente
pareto_fronts_cache = {}

def get_cached_pareto_front(problem_name, m):
    key = (problem_name, m)
    if key in pareto_fronts_cache:
        return pareto_fronts_cache[key]
    
    ref_front = None
    ref_front_path = f"resources/ReferenceFronts/DTLZ/{problem_name}.{m}D.csv"
    if os.path.exists(ref_front_path):
        try:
            ref_front = np.loadtxt(ref_front_path, delimiter=",")
        except Exception:
            pass
    pareto_fronts_cache[key] = ref_front
    return ref_front

def compute_row_igd(row):
    label = row["label"]
    algo = row["algorithm"]
    mode = row["mode"]
    seed = row["seed"]
    problem_name = row["problem"]
    m = row["m"]
    
    # Caminhos possíveis para o arquivo da população resultante da execução
    paths = [
        f"out/{label}_{algo}_{mode}_seed_{seed}_population.csv",
        f"out/{label}_{mode}_seed_{seed}_population.csv"
    ]
    
    pop_path = None
    for p in paths:
        if os.path.exists(p):
            pop_path = p
            break
            
    if pop_path is None:
        # Caso a população específica não seja encontrada, usamos o IGD original da tabela
        return row.get("igd", np.nan)
        
    try:
        from src.QualityIndicator import IGD
        pop_df = pd.read_csv(pop_path)
        obj_cols = [c for c in pop_df.columns if c.startswith("obj_")]
        if not obj_cols:
            return row.get("igd", np.nan)
        front = pop_df[obj_cols].values
        
        ref_front = get_cached_pareto_front(problem_name, m)
        if ref_front is None:
            return row.get("igd", np.nan)
            
        # Normalização: divide os objetivos por 0.5 para DTLZ1 e por 1.0 para os outros problemas
        norm_factor = 0.5 if problem_name == "DTLZ1" else 1.0
        front_normalized = front / norm_factor
        ref_front_normalized = ref_front / norm_factor
        
        indicator_igd = IGD(ref_front_normalized.tolist())
        return indicator_igd.calculate(front_normalized.tolist())
    except Exception:
        return row.get("igd", np.nan)

### Leitura dos Dados e Pré-processamento

Lemos o arquivo `out/all_results.csv` e aplicamos o cálculo de IGD normalizado para todas as execuções. Além disso, mapeamos os nomes internos dos algoritmos para nomes amigáveis para exibição.

In [32]:
csv_path = "out/all_results.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"O arquivo {csv_path} não foi encontrado. Execute os experimentos primeiro.")

df = pd.read_csv(csv_path)
print(f"Foram carregados {len(df)} registros do arquivo CSV.")

# O IGD já está calculado no all_results.csv e é idêntico ao recalculado.
# Evitamos a recalculação lenta lendo milhares de arquivos.
df["igd_normalized"] = df["igd"]

# Mapeamento de nomes de algoritmos
def map_algorithm_name(row):
    algo = row["algorithm"]
    mode = row["mode"]
    
    name_map = {
        "MOEAD": "MOEA/D",
        "NSGAII": "NSGA-II",
        "NSGAIII": "NSGA-III"
    }
    algo_name = name_map.get(algo, algo)
    
    if mode == "dvl_framework":
        return f"{algo_name} + DVL"
    else:
        return algo_name

df["display_algorithm"] = df.apply(map_algorithm_name, axis=1)

Foram carregados 4919 registros do arquivo CSV.


### Geração das Tabelas Resumidas por Algoritmo

Para cada algoritmo, agrupamos as execuções por **Problema**, número de **Objetivos (M)** e quantidade máxima de **Avaliações**. A seguir, geramos uma tabela agregada e exibimos em HTML (tabela interativa do Pandas) e também imprimimos uma versão em formato Markdown (para fácil cópia e uso externo).

In [33]:
# Função personalizada para converter o DataFrame em Markdown (sem depender da biblioteca tabulate)
def to_markdown_custom(df_table):
    headers = df_table.columns.tolist()
    header_row = "| " + " | ".join(map(str, headers)) + " |"
    separator_row = "| " + " | ".join(["---"] * len(headers)) + " |"
    data_rows = []
    for idx, row in df_table.iterrows():
        data_rows.append("| " + " | ".join(map(lambda x: str(x).replace("\n", " "), row.values)) + " |")
    return "\n".join([header_row, separator_row] + data_rows)

def generate_summary_table(algo_df):
    if algo_df.empty:
        return pd.DataFrame()
    grouped = algo_df.groupby(["problem", "m", "max_evaluations"])
    
    rows = []
    for (problem, m, max_eval), group in grouped:
        # Hypervolume
        hv_mean = group["hypervolume"].mean()
        hv_std = group["hypervolume"].std()
        
        # IGD (usa igd_normalized se disponível, senão igd original)
        igd_vals = group["igd_normalized"].dropna()
        if len(igd_vals) == 0:
            igd_vals = group["igd"].dropna()
        
        igd_mean = igd_vals.mean() if len(igd_vals) > 0 else np.nan
        igd_std = igd_vals.std() if len(igd_vals) > 0 else np.nan
        
        # Tempo de Execução
        time_mean = group["cpu_time_seconds"].mean()
        time_std = group["cpu_time_seconds"].std()
        
        # Sucesso
        success_count = (group["status"] == "success").sum()
        total_count = len(group)
        
        # Formatação das strings
        def fmt_val(mean, std):
            if pd.isna(mean):
                return "N/A"
            if pd.isna(std):
                std = 0.0
            return f"{mean:.6f} ± {std:.6f}"
            
        hv_str = fmt_val(hv_mean, hv_std)
        igd_str = fmt_val(igd_mean, igd_std)
        time_str = fmt_val(time_mean, time_std)
        success_str = f"{success_count}/{total_count}"
        
        rows.append({
            "Problema": problem,
            "Objetivos (M)": m,
            "Avaliações": max_eval,
            "Hypervolume": hv_str,
            "IGD": igd_str,
            "Tempo de Execução (s)": time_str,
            "Sucesso": success_str
        })
        
    summary_table = pd.DataFrame(rows)
    if not summary_table.empty:
        summary_table = summary_table.sort_values(by=["Problema", "Objetivos (M)", "Avaliações"])
    return summary_table

base_algorithms = ["MOEA/D", "NSGA-II", "NSGA-III"]

for base_algo in base_algorithms:
    # Exibe o título principal do Algoritmo
    display(HTML(f"<h3 style='color: #1a5f7a; margin-top: 40px; font-weight: bold; border-bottom: 2px solid #1a5f7a; padding-bottom: 5px;'>Resultados: {base_algo}</h3>"))
    
    # Gerar tabelas
    df_pure_algo = df[df["display_algorithm"] == base_algo]
    df_dvl_algo = df[df["display_algorithm"] == f"{base_algo} + DVL"]
    
    table_pure = generate_summary_table(df_pure_algo)
    table_dvl = generate_summary_table(df_dvl_algo)
    
    # Estilização CSS com cores de texto explícitas para garantir compatibilidade com temas escuros
    table_style = """
    <style>
        .custom-table-container {
            display: flex;
            gap: 20px;
            width: 100%;
            overflow-x: auto;
            margin-top: 15px;
            margin-bottom: 15px;
        }
        .custom-table-wrapper {
            flex: 1;
            min-width: 45%;
            border: 1px solid #ddd;
            border-radius: 8px;
            padding: 15px;
            background-color: #ffffff !important;
            color: #222222 !important;
            box-shadow: 0 4px 6px rgba(0,0,0,0.05);
        }
        .custom-table-wrapper h4 {
            color: #1a5f7a !important;
            font-weight: bold;
            margin-top: 0;
            margin-bottom: 15px;
            border-bottom: 1px solid #eee;
            padding-bottom: 8px;
        }
        .custom-table-wrapper p {
            color: #666666 !important;
        }
        .custom-table-wrapper table {
            width: 100% !important;
            border-collapse: collapse;
            margin: 0 !important;
            color: #222222 !important;
        }
        .custom-table-wrapper th {
            background-color: #1a5f7a !important;
            color: white !important;
            font-weight: bold;
            padding: 8px !important;
            text-align: left !important;
        }
        .custom-table-wrapper td {
            padding: 6px 8px !important;
            border-bottom: 1px solid #eee !important;
            text-align: left !important;
            color: #222222 !important;
        }
        .custom-table-wrapper tr:hover {
            background-color: #f5f5f5 !important;
        }
    </style>
    """
    
    html_pure = f"<h4>Sem DVL (MOEA Puro)</h4>{table_pure.to_html(index=False)}" if not table_pure.empty else "<h4>Sem DVL (MOEA Puro)</h4><p>Sem dados.</p>"
    html_dvl = f"<h4>Com DVL</h4>{table_dvl.to_html(index=False)}" if not table_dvl.empty else "<h4>Com DVL</h4><p>Sem dados.</p>"
    
    flex_container = f"""
    {table_style}
    <div class="custom-table-container">
        <div class="custom-table-wrapper">
            {html_pure}
        </div>
        <div class="custom-table-wrapper">
            {html_dvl}
        </div>
    </div>
    """
    display(HTML(flex_container))
    
    # if not table_pure.empty:
    #     print(f"\n### Tabela em Markdown para {base_algo} (Sem DVL):")
    #     print(to_markdown_custom(table_pure))
    # if not table_dvl.empty:
    #     print(f"\n### Tabela em Markdown para {base_algo} + DVL:")
    #     print(to_markdown_custom(table_dvl))
    # print("\n" + "="*80 + "\n")


Problema,Objetivos (M),Avaliações,Hypervolume,IGD,Tempo de Execução (s),Sucesso
DTLZ1,3,250,0.000000 ± 0.000000,183.947746 ± 26.792143,0.040417 ± 0.005051,26/26
DTLZ1,3,500,0.000000 ± 0.000000,151.171535 ± 17.576552,0.077960 ± 0.004295,24/24
DTLZ1,3,1000,0.000000 ± 0.000000,139.155399 ± 16.376470,0.151175 ± 0.004560,24/24
DTLZ1,3,1500,0.000000 ± 0.000000,128.491127 ± 15.991014,0.227191 ± 0.005916,24/24
DTLZ1,3,10000,0.000000 ± 0.000000,83.208651 ± 13.001375,1.504537 ± 0.040234,23/23
DTLZ1,10,250,0.850013 ± 0.105759,1.831095 ± 0.942199,0.090961 ± 0.009971,20/20
DTLZ1,10,500,0.790735 ± 0.289902,3.390739 ± 2.622853,0.132758 ± 0.010183,20/20
DTLZ1,10,1000,0.930483 ± 0.163038,2.165226 ± 2.107059,0.209406 ± 0.010017,20/20
DTLZ1,10,1500,0.957211 ± 0.101164,1.890831 ± 1.887114,0.285248 ± 0.003033,20/20
DTLZ1,10,10000,0.994797 ± 0.015735,1.017944 ± 1.162021,1.578670 ± 0.033663,20/20


Problema,Objetivos (M),Avaliações,Hypervolume,IGD,Tempo de Execução (s),Sucesso
DTLZ1,3,250,0.000000 ± 0.000000,257.286269 ± 45.887621,0.060109 ± 0.001942,26/26
DTLZ1,3,500,0.000000 ± 0.000000,176.563159 ± 20.220020,0.148048 ± 0.010985,24/24
DTLZ1,3,1000,0.000000 ± 0.000000,141.181756 ± 16.668306,0.308612 ± 0.010338,24/24
DTLZ1,3,1500,0.000000 ± 0.000000,134.712047 ± 12.303229,0.482517 ± 0.011788,24/24
DTLZ1,3,10000,0.000000 ± 0.000000,94.940749 ± 10.752893,3.312482 ± 0.029433,22/22
DTLZ1,10,250,0.933054 ± 0.036427,1.096839 ± 0.354711,0.055399 ± 0.004583,20/20
DTLZ1,10,500,0.944241 ± 0.035372,1.089529 ± 0.361989,0.219713 ± 0.003052,20/20
DTLZ1,10,1000,0.993856 ± 0.007477,0.890579 ± 0.267870,0.803622 ± 0.017727,20/20
DTLZ1,10,1500,0.999126 ± 0.001209,0.770623 ± 0.170581,1.253591 ± 0.019501,20/20
DTLZ1,10,10000,0.999931 ± 0.000114,0.634154 ± 0.069655,9.473639 ± 0.108542,20/20


Problema,Objetivos (M),Avaliações,Hypervolume,IGD,Tempo de Execução (s),Sucesso
DTLZ1,3,250,0.342536 ± 0.207148,1.069197 ± 0.430075,0.104793 ± 0.013511,25/25
DTLZ1,3,500,0.629379 ± 0.141687,0.606241 ± 0.177873,0.244881 ± 0.015919,24/24
DTLZ1,3,1000,0.807656 ± 0.204252,0.543941 ± 0.129437,0.495027 ± 0.015708,24/24
DTLZ1,3,1500,0.956073 ± 0.099356,0.514575 ± 0.095238,0.771360 ± 0.026459,24/24
DTLZ1,3,10000,0.975918 ± 0.075900,0.533300 ± 0.086317,4.988703 ± 0.163125,22/22
DTLZ1,10,250,0.943487 ± 0.032506,1.092284 ± 0.353384,0.373396 ± 0.020452,20/20
DTLZ1,10,500,0.965420 ± 0.019897,1.081852 ± 0.358674,0.834169 ± 0.065563,20/20
DTLZ1,10,1000,0.995575 ± 0.010054,0.852022 ± 0.186642,1.779765 ± 0.111561,20/20
DTLZ1,10,1500,0.999847 ± 0.000154,0.692313 ± 0.162058,2.820797 ± 0.112935,20/20
DTLZ1,10,10000,1.000000 ± 0.000000,0.387955 ± 0.022441,20.795899 ± 0.481550,20/20
